# The Ninefold Tide — Original AI voice cast, first audition run

This private Kaggle notebook runs the real free Qwen3-TTS 1.7B VoiceDesign and Base models to produce original voice references for Naya, Kito and Iri, then three candidates for each pilot dialogue line and the canonical name drills. It does not impersonate any real actor. All voices are unapproved until someone listens to them.

Speech model caches go to temporary Kaggle storage. Only WAV/JSON outputs and small logs are saved. Never merge voice audition files into the public code repository.


In [ ]:
import os, sys, subprocess, json, pathlib, time, traceback, zipfile, hashlib
from IPython.display import display, Audio
from datetime import datetime, timezone

# This Kaggle session is private. Never copy model weights to /kaggle/working.
os.environ["HF_HOME"] = "/kaggle/temp/ninefold_hf_cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
root = pathlib.Path("/kaggle/temp/ninefold_tools")
out = pathlib.Path("/kaggle/working/ninefold_voice_ep101")
out.mkdir(parents=True, exist_ok=True)
status_file = out / "voice_run_status.json"
record = {"project":"The Ninefold Tide", "series":"ep101",
          "cast":["naya","kito","iri"], "approval":"UNAPPROVED AUDITIONS",
          "state":"STARTED","completed_stages":[],
          "start_utc":datetime.now(timezone.utc).isoformat()}
def save_status():
    status_file.write_text(json.dumps(record, indent=2), encoding="utf-8")
def run_step(name, argv, timeout_seconds):
    print("START:", name, flush=True)
    start = time.time()
    proc = subprocess.run(argv, text=True, capture_output=True, timeout=timeout_seconds)
    log = out / ("step_" + name.replace(" ","_") + ".log")
    log.write_text((proc.stdout or "")[-25000:] + "\n--- STDERR ---\n" + (proc.stderr or "")[-25000:],encoding="utf-8")
    print("END:", name, "exit", proc.returncode, "seconds", round(time.time()-start, 1),flush=True)
    if proc.returncode != 0:
        print("ERROR LOG (last 2200 chars):", (proc.stderr or proc.stdout or "")[-2200:],flush=True)
        raise RuntimeError(f"Step {name} failed with exit code {proc.returncode}")
    record["completed_stages"].append(name)
    save_status()
save_status()
try:
    run_step("GPU preflight", ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], 40)
    run_step("Install verified Qwen inference package",
             [sys.executable,"-m","pip","install","-q","qwen-tts","soundfile"], 700)
    run_step("Fetch exact production pipeline",
             ["git","clone","-q","--depth","1","https://github.com/eghosa001/TOOLS.git",str(root)], 140)
    worker = root / "video/ninefold_tide/audio/qwen_audio_worker.py"
    control = root / "video/ninefold_tide/audio/voice_pipeline.py"
    run_step("Prepare voice workspace",
             [sys.executable,str(control),"prepare","--workspace",str(out)], 30)
    run_step("Confirm cast and screenplay",
             [sys.executable,str(control),"audit","--stage","structure","--workspace",str(out)], 40)
    run_step("Check 16GB-class GPU",
             [sys.executable,str(worker),"gpu-check","--workspace",str(out)], 60)
    # Qwen VoiceDesign 1.7B: original voices. Will download model on Kaggle, not phone.
    run_step("Generate three original voice references",
             [sys.executable,str(worker),"design","--characters","naya,kito,iri",
              "--workspace",str(out),"--allow-download"], 1350)
    refs={char:str(out/"references"/f"{char}_unapproved.wav") for char in ("naya","kito","iri")}
    if not all(pathlib.Path(p).is_file() for p in refs.values()):
        raise RuntimeError("No complete trio of voice references generated")
    # Qwen Base 1.7B: takes from original synthetic anchors; model loaded separately.
    run_step("Generate pilot dialogue and pronunciation takes",
             [sys.executable,str(worker),"perform","--characters","naya,kito,iri",
              "--takes-per-line","3","--name-drills","--workspace",str(out),
              "--allow-download"], 1700)
    reference_paths=[pathlib.Path(p) for p in refs.values()]
    take_paths=list((out/"takes").glob("*.wav"))
    record["reference_count"]=len(reference_paths)
    record["dialogue_and_name_take_count"]=len(take_paths)
    record["state"]="AUDITIONS_GENERATED_UNAPPROVED" if len(take_paths) >= 21 else "PARTIAL_AUDITIONS"
except Exception as exc:
    record["state"]="PARTIAL_OR_FAILED"
    record["error"]=str(exc)[:1200]
    print("PIPELINE STOPPED:", str(exc),flush=True)
finally:
    record["end_utc"]=datetime.now(timezone.utc).isoformat()
    record["reference_wavs"]=[p.name for p in (out/"references").glob("*.wav")]
    record["candidate_wavs"]=[p.name for p in (out/"takes").glob("*.wav")]
    save_status()
    archive=pathlib.Path("/kaggle/working/ninefold_voice_ep101_UNAPPROVED.zip")
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for path in out.rglob("*"):
            if path.is_file() and path.suffix.lower() in (".wav",".json",".log"):
                z.write(path,arcname=path.relative_to(out))
    print("Final status:",record["state"],flush=True)
    print("Archive:",archive.name,"bytes:",archive.stat().st_size,flush=True)
    print("This notebook produces AUDITION CANDIDATES ONLY; no voices are claimed approved.",flush=True)

if record["state"] not in ("AUDITIONS_GENERATED_UNAPPROVED",):
    print("The run did not complete every take. See voice_run_status.json and step logs.",flush=True)
